In [1]:
from bs4 import BeautifulSoup
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import nltk
from nltk.tokenize import word_tokenize
import os
import tqdm
import numpy as np
import gc

In [ ]:
!pip show torch
!pip show torchtext

In [ ]:
nltk.download('punkt')

In [2]:
regex = re.compile(r"\((\d+), '([^']+)', '([^']+)', (\d+), ('[^']+'\))")

mapper = dict()

with open('n96ncsr5g4-1/index.txt', 'r') as file:
    fields = file.readlines()
    for field in fields:
        match = re.match(regex, field)
        if match:
            mapper[match.group(3)] = match.group(4)

print(len(mapper))


79987


In [3]:
files = []

for part in range(1, 7):  # Dataset parts 1 to 6
    dataset_path = f'n96ncsr5g4-1/dataset/dataset-part-{part}'
    for file in os.listdir(dataset_path):
        full_path = f'{dataset_path}/{file}'
        files.append(full_path)

X = []
y = []
max_files = 10000

for file in tqdm.tqdm(files[:max_files]):
    # Extract the filename from the full path
    filename = file.split('/')[-1]
    if filename in mapper:
        with open(file, 'r') as file:
            html_content = file.read()
            soup = BeautifulSoup(html_content, 'html.parser')
            text = soup.get_text()
            text = text.strip()
            X.append(text)
            y.append(mapper[filename])

100%|██████████| 10000/10000 [02:09<00:00, 77.52it/s]


### Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# Create a list to store the analysis results
analysis_results = []

for i, (text, label) in enumerate(zip(X, y)):
    # Count words and characters
    words = word_tokenize(text)
    chars = list(text)
    
    # Count unique words and characters
    unique_words = set(words)
    unique_chars = set(chars)
    
    # Store the results
    analysis_results.append({
        'File Index': i,
        'Label': label,
        'Word Count': len(words),
        'Character Count': len(chars),
        'Unique Word Count': len(unique_words),
        'Unique Character Count': len(unique_chars)
    })

# Convert to DataFrame for easy analysis
df = pd.DataFrame(analysis_results)

# Display summary statistics
print(df.describe())

# Plot histograms
fig, axs = plt.subplots(2, 2, figsize=(15, 15))
df['Word Count'].hist(ax=axs[0, 0], bins=20)
axs[0, 0].set_title('Distribution of Word Counts')
df['Character Count'].hist(ax=axs[0, 1], bins=20)
axs[0, 1].set_title('Distribution of Character Counts')
df['Unique Word Count'].hist(ax=axs[1, 0], bins=20)
axs[1, 0].set_title('Distribution of Unique Word Counts')
df['Unique Character Count'].hist(ax=axs[1, 1], bins=20)
axs[1, 1].set_title('Distribution of Unique Character Counts')
plt.tight_layout()
plt.show()

# Display top 10 most common words and characters across all files
all_words = [word for text in X for word in text.split()]
all_chars = [char for text in X for char in list(text)]

print("\nTop 10 most common words:")
print(Counter(all_words).most_common(10))

print("\nTop 10 most common characters:")
print(Counter(all_chars).most_common(10))

# Calculate and display average word length
avg_word_length = sum(len(word) for word in all_words) / len(all_words)
print(f"\nAverage word length: {avg_word_length:.2f} characters")


In [ ]:
# Analyze non-alphanumeric characters in files
non_alphanumeric_analysis = []

for i, text in enumerate(X):
    non_alphanumeric = set(char for char in text if not char.isalnum() and not char.isspace())
    if non_alphanumeric:
        non_alphanumeric_analysis.append({
            'File Index': i,
            'Non-alphanumeric Characters': list(non_alphanumeric)
        })

# List all unique non-alphanumeric characters
all_non_alphanumeric = set(char for item in non_alphanumeric_analysis for char in item['Non-alphanumeric Characters'])
print("All non-alphanumeric characters found:")
print(', '.join(sorted(all_non_alphanumeric)))


In [ ]:
# Analyze unique words across all files
all_unique_words = set(word for text in X for word in word_tokenize(text))

print(f"\nTotal number of unique words across all files: {len(all_unique_words)}")

# Display some statistics about word lengths
word_lengths = [len(word) for word in all_unique_words]
avg_word_length = sum(word_lengths) / len(word_lengths)
max_word_length = max(word_lengths)
min_word_length = min(word_lengths)

print(f"Average length of unique words: {avg_word_length:.2f} characters")
print(f"Longest word length: {max_word_length} characters")
print(f"Shortest word length: {min_word_length} characters")

# Display distribution of word lengths
plt.figure(figsize=(10, 5))
plt.hist(word_lengths, bins=range(min_word_length, max_word_length + 2, 1))
plt.title('Distribution of Unique Word Lengths')
plt.xlabel('Word Length')
plt.ylabel('Frequency')
plt.show()

# Display some examples of long and short words
long_words = [word for word in all_unique_words if len(word) == max_word_length]
short_words = [word for word in all_unique_words if len(word) == min_word_length]

print(f"\nExamples of longest words (length {max_word_length}):")
print(', '.join(long_words[:10]))  # Display up to 10 examples

print(f"\nExamples of shortest words (length {min_word_length}):")
print(', '.join(short_words[:10]))  # Display up to 10 examples


In [4]:
# Analyze non-ASCII words and their frequencies
from collections import Counter
import unicodedata

def is_ascii(s):
    return all(ord(c) < 128 for c in s)

non_ascii_words = [word for text in X for word in word_tokenize(text) if not is_ascii(word)]
non_ascii_word_freq = Counter(non_ascii_words)

print(f"Total number of unique non-ASCII words: {len(non_ascii_word_freq)}")

# Sort non-ASCII words by frequency (descending order)
sorted_non_ascii_words = sorted(non_ascii_word_freq.items(), key=lambda x: x[1], reverse=True)

print("\nTop 20 most frequent non-ASCII words and their frequencies:")
for word, freq in sorted_non_ascii_words[:20]:
    print(f"{word}: {freq}")

# Function to get Unicode category for a character
def get_unicode_category(char):
    return unicodedata.category(char)

# Analyze Unicode categories of non-ASCII characters
non_ascii_char_categories = Counter()
for word in non_ascii_words:
    for char in word:
        if not is_ascii(char):
            non_ascii_char_categories[get_unicode_category(char)] += 1

print("\nUnicode categories of non-ASCII characters:")
for category, count in non_ascii_char_categories.most_common():
    print(f"{category}: {count}")


Total number of unique non-ASCII words: 126322

Top 20 most frequent non-ASCII words and their frequencies:
’: 33609
“: 10686
”: 10684
–: 9649
»: 5690
©: 4900
•: 4490
‘: 4364
·: 4250
—: 3901
в: 3722
и: 3493
«: 3404
„: 2813
на: 2342
×: 2145
с: 1552
…: 1545
не: 1411
для: 1309

Unicode categories of non-ASCII characters:
Ll: 998380
Lo: 226282
Lu: 140467
Pf: 52779
Po: 45514
So: 41730
Pd: 26051
Pi: 20857
No: 18659
Sm: 16933
Sc: 16715
Sk: 14276
Cc: 11752
Cf: 9294
Ps: 7475
Lm: 6770
Mn: 6557
Mc: 5435
Pe: 2036
Co: 639
Nd: 57
Nl: 11
Me: 7
Pc: 3


### Vocab Builder from Iterator

In [5]:
# Based on torchtext build_vocab_from_iterator function

from collections import Counter, defaultdict

class Vocab:
    """A simple vocabulary object to map tokens to numerical identifiers."""

    UNK = '<unk>'

    def __init__(self, counter, max_size=None, min_freq=1, specials=('<unk>', '<pad>')):
        """Initialize the Vocab object from a counter of token frequencies."""
        self.freqs = counter
        self.itos = list(specials)  # Initialize with special tokens
        self.unk_index = self.itos.index(Vocab.UNK) if Vocab.UNK in specials else None
        min_freq = max(min_freq, 1)

        # Remove special tokens from the counter
        for special in specials:
            del counter[special]

        # Sort by frequency (descending), then alphabetically
        words_and_frequencies = sorted(counter.items(), key=lambda tup: (-tup[1], tup[0]))

        # Add tokens to the vocabulary, respecting max_size and min_freq
        for word, freq in words_and_frequencies:
            if freq < min_freq or (max_size is not None and len(self.itos) >= max_size):
                break
            self.itos.append(word)

        # Create the stoi mapping (string-to-index)
        self.stoi = defaultdict(lambda: self.unk_index, {tok: i for i, tok in enumerate(self.itos)})

    def set_default_index(self, index):
        """Set the default index to use for unknown tokens."""
        self.unk_index = index
        # Update the defaultdict to use the new default index
        self.stoi = defaultdict(lambda: self.unk_index, self.stoi)

    def __getitem__(self, token):
        """Return the index of the token or the default index if not found."""
        return self.stoi.get(token, self.unk_index)

    def __len__(self):
        return len(self.itos)

    def lookup_indices(self, tokens):
        """Convert a list of tokens to their respective indices."""
        return [self[token] for token in tokens]

    def lookup_tokens(self, indices):
        """Convert a list of indices back to their respective tokens."""
        return [self.itos[idx] for idx in indices]

def build_vocab_from_iterator(iterator, max_size=None, min_freq=1, specials=('<unk>', '<pad>')):
    """
    Build a Vocab from an iterator of tokenized sequences.
    
    Args:
        iterator: An iterable that yields tokenized sequences.
        max_size: Maximum size of the vocabulary. If None, no limit. Default: None.
        min_freq: Minimum frequency for a token to be included in the vocabulary. Default: 1.
        specials: Special tokens to be added to the vocabulary. Default: ('<unk>', '<pad>').
    
    Returns:
        vocab: A Vocab object.
    """
    counter = Counter()
    for tokens in iterator:
        counter.update(tokens)  # Update the frequency count for each token
    
    return Vocab(counter, max_size=max_size, min_freq=min_freq, specials=specials)

"""
# Example usage:
token_sequences = [['this', 'is', 'a', 'sentence'], ['another', 'sentence', 'here']]

# Build the vocabulary
vocab = build_vocab_from_iterator(token_sequences)

# Set the default index to the index of <unk>
vocab.set_default_index(vocab.stoi['<unk>'])

# Print vocab mappings
print("String-to-Index Mapping:", vocab.stoi)
print("Index-to-String Mapping:", vocab.itos)

# Lookup token indices
print("Indices for ['this', 'is', 'missing']:", vocab.lookup_indices(['this', 'is', 'missing']))
"""


'\n# Example usage:\ntoken_sequences = [[\'this\', \'is\', \'a\', \'sentence\'], [\'another\', \'sentence\', \'here\']]\n\n# Build the vocabulary\nvocab = build_vocab_from_iterator(token_sequences)\n\n# Set the default index to the index of <unk>\nvocab.set_default_index(vocab.stoi[\'<unk>\'])\n\n# Print vocab mappings\nprint("String-to-Index Mapping:", vocab.stoi)\nprint("Index-to-String Mapping:", vocab.itos)\n\n# Lookup token indices\nprint("Indices for [\'this\', \'is\', \'missing\']:", vocab.lookup_indices([\'this\', \'is\', \'missing\']))\n'

### Preprocessing

In [6]:
# Tokenization (character-level and word-level)
def tokenize_char(doc):
    return list(doc)  # Character-level tokenization

def tokenize_char_ascii(doc):
    chars = tokenize_char(doc)
    return [char for char in chars if is_ascii(char)]

def tokenize_word(doc):
    return word_tokenize(doc)  # Word-level tokenization

def tokenize_word_ascii(text):
    # Tokenize the text into words
    words = word_tokenize(text)
    # Filter out non-ASCII words
    ascii_words = [word for word in words if is_ascii(word)]
    return ascii_words

char_sequences = [tokenize_char_ascii(_text) for _text in X]
#word_sequences = [tokenize_word_ascii(_text) for _text in X] 

# Building the vocab from iterator
def yield_tokens(data_iter):
    for seq in data_iter:
        yield seq

char_vocab = build_vocab_from_iterator(yield_tokens(char_sequences), specials=["<unk>"])
#word_vocab = build_vocab_from_iterator(yield_tokens(word_sequences), specials=["<unk>"])

# Setting default index for unknown tokens
char_vocab.set_default_index(char_vocab["<unk>"])
#word_vocab.set_default_index(word_vocab["<unk>"])

# Check vocab
# print("Character Vocabulary:", char_vocab.get_stoi())  # String-to-index mapping
# print("Word Vocabulary:", word_vocab.get_stoi())  # String-to-index mapping


Counter({' ': 54466281, '\n': 11615946, 'e': 4823671, 'a': 3266953, 't': 3256366, 'o': 3210094, 'i': 2987821, 'n': 2871620, 'r': 2737324, 's': 2661253, 'l': 1703009, 'c': 1419131, 'd': 1387242, 'h': 1288403, 'u': 1248677, 'm': 996677, 'p': 907294, 'g': 811477, 'y': 727070, 'f': 690484, '.': 542064, 'w': 539276, 'b': 534277, 'v': 462708, 'S': 386049, ',': 372386, 'k': 352919, '0': 328184, 'C': 319393, 'A': 309392, 'T': 285072, '1': 280293, 'P': 262268, '2': 249510, '-': 233468, 'I': 226032, 'M': 223285, 'D': 186629, 'R': 186217, 'E': 183478, 'B': 174776, 'L': 156032, 'F': 150555, 'N': 150200, 'O': 138972, ':': 129617, 'W': 121877, '"': 120797, 'H': 119173, 'x': 118650, '3': 116251, 'G': 114105, '5': 110778, '9': 107562, ')': 105555, '(': 104535, '/': 103247, '4': 102277, 'U': 88287, '6': 88108, '8': 85312, '7': 80883, 'V': 76701, 'z': 70686, 'q': 60300, "'": 58917, 'j': 53073, 'J': 52036, 'Y': 48065, 'K': 46594, '&': 45038, '?': 33516, '_': 30358, ']': 29370, '[': 29259, '=': 28770, '>'

In [8]:
y = list(map(int, y))

In [9]:
import torch
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

def sliding_window(input_sequence, window_size, stride):
    """
    Splits an input sequence into overlapping windows.
    
    Args:
        input_sequence (list or tensor): The input sequence (tokenized document).
        window_size (int): The size of each window (number of tokens per window).
        stride (int): The step size between windows.

    Returns:
        windows (list of tensors): List of tensor windows.
    """
    sequence_length = len(input_sequence)
    windows = []

    # Generate windows using sliding window 
    for i in range(0, sequence_length - window_size + 1, stride):
        window = input_sequence[i:i + window_size]
        windows.append(torch.tensor(window))  # Convert to tensor

    # Optionally, handle the last window if there are remaining tokens
    if sequence_length % stride != 0:
        last_window = input_sequence[-window_size:]  # Grab the last window
        windows.append(torch.tensor(last_window))
    
    return windows

# Parameters
window_size = 512
stride = 256

# char_sequences is already tokenized and y contains the labels
char_indices = [[char_vocab[char] for char in seq] for seq in char_sequences]
char_indices = [torch.tensor(seq) for seq in char_indices]

# Split the data into training and test sets
train_char_indices, test_char_indices, train_labels, test_labels = train_test_split(
    char_indices, y, test_size=0.2, random_state=42)

# Apply sliding window to training data
train_char_windows = [sliding_window(seq, window_size, stride) for seq in train_char_indices]
train_windows_flat = [window for doc_windows in train_char_windows for window in doc_windows]

# Apply sliding window to test data
test_char_windows = [sliding_window(seq, window_size, stride) for seq in test_char_indices]
test_windows_flat = [window for doc_windows in test_char_windows for window in doc_windows]

# Apply document-level labeling to each window (replicate label for each window in training set)
train_label_windows = []
for doc_label, doc_windows in zip(train_labels, train_char_windows):
    train_label_windows.extend([doc_label] * len(doc_windows))

# Apply document-level labeling to each window (replicate label for each window in test set)
test_label_windows = []
for doc_label, doc_windows in zip(test_labels, test_char_windows):
    test_label_windows.extend([doc_label] * len(doc_windows))

# Convert label windows to tensors
train_label_windows_tensor = torch.tensor(train_label_windows, dtype=torch.float32)
test_label_windows_tensor = torch.tensor(test_label_windows, dtype=torch.float32)

# Optionally, pad each window to the same size for both training and testing
train_char_padded_windows = pad_sequence(train_windows_flat, batch_first=True, padding_value=0)
test_char_padded_windows = pad_sequence(test_windows_flat, batch_first=True, padding_value=0)

# Now, train_char_padded_windows and test_char_padded_windows can be used for training and testing
print(train_char_padded_windows.shape)
print(train_label_windows_tensor.shape)
print(test_char_padded_windows.shape)
print(test_label_windows_tensor.shape)


/var/folders/lz/54h48fsx6294gn68v0q9pp4m0000gn/T/ipykernel_6464/2123168317.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  windows.append(torch.tensor(window))  # Convert to tensor
/var/folders/lz/54h48fsx6294gn68v0q9pp4m0000gn/T/ipykernel_6464/2123168317.py:28: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  windows.append(torch.tensor(last_window))


torch.Size([344402, 512])
torch.Size([344402])
torch.Size([94594, 512])
torch.Size([94594])


In [ ]:
# Analyze the length of tokenized sequences before padding
char_lengths = [len(seq) for seq in char_sequences]

# Print the lengths of the sequences
for i, length in enumerate(char_lengths):
    print(f"Document {i+1} has {length} characters (before padding).")

padded_lengths = [len(seq) for seq in train_char_padded_windows]

# Print the lengths of the sequences after padding
for i, length in enumerate(padded_lengths):
    print(f"Document {i+1} has {length} characters (after padding).")

print(f"Minimum length: {np.min(char_lengths)}")
print(f"Maximum length: {np.max(char_lengths)}")
print(f"Average length: {np.mean(char_lengths)}")


In [27]:
"""
embedding_dim = 5

# Embedding layers
char_embedding = nn.Embedding(len(char_vocab), embedding_dim)
#word_embedding = nn.Embedding(len(word_vocab), embedding_dim)

char_embedded = char_embedding(char_padded_windows)
#word_embedded = word_embedding(word_padded)

# Concatenate embeddings
#concatenated = torch.cat((char_embedded, word_embedded), dim=1)

# concatenated = char_embedded
# print(concatenated.shape)
"""

In [10]:
class HTMLPhishCNN(nn.Module):
    def __init__(self, embedding_dim, vocab_size):
        super(HTMLPhishCNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Conv layers with 1 kernel size for now, scale up later
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=32, kernel_size=k) for k in [3]
        ])
        
        # Max Pooling layer
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # Placeholder for fully connected layer; dynamically set later
        self.fc = None
        
        # Output layer for binary classification
        self.output = nn.Linear(10, 1)
    
    def forward(self, x):

        x = self.embedding(x)
        # Apply Conv1D and ReLU to each layer
        x = [F.relu(conv(x.permute(0, 2, 1))) for conv in self.conv_layers]
        
        # Apply Max Pooling to each output
        x = [self.pool(conv) for conv in x]
        
        # Concatenate the output of all convolution layers along the sequence length
        x = torch.cat(x, dim=2)
        
        # Dynamically calculate the flattened size
        if self.fc is None:
            flatten_size = x.view(x.size(0), -1).size(1)
            self.fc = nn.Linear(flatten_size, 10)
        
        # Flatten the output
        x = x.view(x.size(0), -1)
        
        # Fully connected layer
        x = F.relu(self.fc(x))
        
        # Output layer (sigmoid for regression)
        x = torch.sigmoid(self.output(x))
        
        return x


In [ ]:
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    print("MPS is available and can be used for training.")
else:
    print("MPS is not available.")

In [11]:
# Check if MPS (Apple GPU API since I'm using a Mac) is available, otherwise fall back to CPU
go_ahead = False
device = torch.device("mps" if torch.backends.mps.is_available() and go_ahead else "cpu")
print(f"Using device: {device}")

# Only move each batch inside the loop to avoid over-allocating memory
char_padded = torch.tensor(train_char_padded_windows, dtype=torch.long)
label_windows_tensor = torch.tensor(train_label_windows_tensor, dtype=torch.float32)

# Create a dataset and dataloader
dataset = TensorDataset(char_padded, label_windows_tensor)
batch_size = 32  # Adjust based on memory constraints
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

embedding_dim = 5
vocab_size = len(char_vocab)
phishcnn = HTMLPhishCNN(embedding_dim, vocab_size)
model = phishcnn.to(device)  # Move model to device

# Define loss function and optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10  # Adjust as needed
for epoch in tqdm.tqdm(range(num_epochs)):
    for batch_char_padded, batch_y in dataloader:
        # Move batch data to whichever inside the loop, batch by batch
        batch_char_padded = batch_char_padded.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        outputs = model(batch_char_padded)
        loss = criterion(outputs.squeeze(), batch_y)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

    # Perform garbage collection after each epoch
    gc.collect()

# Final garbage collection
gc.collect()


Using device: cpu


/var/folders/lz/54h48fsx6294gn68v0q9pp4m0000gn/T/ipykernel_6464/1221845245.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  char_padded = torch.tensor(train_char_padded_windows, dtype=torch.long)
/var/folders/lz/54h48fsx6294gn68v0q9pp4m0000gn/T/ipykernel_6464/1221845245.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label_windows_tensor = torch.tensor(train_label_windows_tensor, dtype=torch.float32)
 10%|█         | 1/10 [01:48<16:13, 108.14s/it]

Epoch [1/10], Loss: 0.1846


 20%|██        | 2/10 [03:45<15:09, 113.71s/it]

Epoch [2/10], Loss: 0.0761


...

 90%|█████████ | 9/10 [17:45<02:00, 120.03s/it]

Epoch [9/10], Loss: 0.2083


100%|██████████| 10/10 [19:46<00:00, 118.65s/it]

Epoch [10/10], Loss: 0.2196


0

In [12]:
# Function to evaluate the model on test data
def evaluate_model(model, test_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():  # Disable gradient computation for testing
        for batch_char_padded, batch_y in test_loader:
            # Move batch data to whichever device
            batch_char_padded = batch_char_padded.to(device)
            batch_y = batch_y.to(device)

            # Forward pass
            outputs = model(batch_char_padded).squeeze()

            # Compute loss
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()

            # Convert outputs to binary predictions (0 or 1)
            predictions = (outputs >= 0.5).float()

            # Count correct predictions
            total_correct += (predictions == batch_y).sum().item()
            total_samples += batch_y.size(0)
    
    avg_loss = total_loss / len(test_loader)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy


test_dataset = TensorDataset(test_char_padded_windows, test_label_windows_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Test the model after training
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Test Loss: 0.2422, Test Accuracy: 0.9230
